# Clase 7 · En vivo: el equipo con firma, y producción

Este cuaderno lo corremos **juntos en clase**: cada sección es una slide de hoy (M7 y M8). Antes de correr cada celda,
**anotá tu predicción**; después comparamos. Si te da distinto que en la slide, también es un resultado.

- Corré las celdas en orden con **Shift + Enter**. La clave: **Secretos** (la llave de la izquierda) → `GOOGLE_API_KEY`.
- Gasta unos 15 pedidos de los 500 del día; la sección 3 (opcional) suma unos 6. Las secciones 5, 7 y 8 no gastan.
- La web acá es **grabada** (siempre la misma respuesta) o **caída** (devuelve un ERROR).
- **No es tu práctica.** Este cuaderno es para mirar y predecir: todo viene escrito. Tu práctica (M7 y M8) la escribís vos, en tu repo o en su propio cuaderno de Colab (el botón de cada práctica en ucu-la).

<details><summary><b>¿Sale un error?</b></summary>

- **«No pude leer el secreto GOOGLE_API_KEY»** o **`API key not valid`:** revisá el nombre del secreto (exacto, en mayúsculas),
  que «Acceso del notebook» esté activado y que la clave esté bien copiada.
- **`503 UNAVAILABLE`** («high demand»): el modelo está saturado, no es tu código. Esperá un minuto y volvé a correr.
- **`429`** (*quota*, *RESOURCE_EXHAUSTED*): se terminó la cuota gratis del día o del minuto. Esperá o seguí mañana.
- **`NameError`:** salteaste una celda o Colab se reinició. Menú *Entorno de ejecución* → *Run before*.
</details>

## 0 · Preparar (una vez, un par de minutos)

In [ ]:
%pip install -q langchain==1.4.6 langchain-google-genai==4.4.2 chromadb==1.5.9 langchain-text-splitters==1.1.3
import os

try:
    from google.colab import userdata                      # en Colab: la clave sale de los Secretos
    os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
except ImportError:
    pass                                                   # fuera de Colab: GOOGLE_API_KEY ya tiene que estar
except Exception as error:                                 # no hay secreto con ese nombre o le falta el acceso
    print("No pude leer el secreto GOOGLE_API_KEY:", type(error).__name__)

from langchain.chat_models import init_chat_model

modelo = init_chat_model("google_genai:gemini-3.5-flash-lite", max_output_tokens=300)  # tope de salida: cuida la cuota
print("Listo." if os.environ.get("GOOGLE_API_KEY") else "Falta la clave: Secretos → GOOGLE_API_KEY.")

In [ ]:
import json
import time
import urllib.request
import uuid
from pathlib import Path
from typing import Literal

import chromadb
import pandas as pd
from langchain.agents import create_agent
from langchain.agents.middleware import ModelCallLimitMiddleware, ToolCallLimitMiddleware
from langchain.tools import tool
from langchain_core.callbacks import BaseCallbackHandler
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langgraph.checkpoint.memory import InMemorySaver

pd.set_option("display.max_colwidth", 80)
# los 5 documentos de ejemplo, las 12 preguntas de selección y la web grabada, de la plantilla del proyecto
PLANTILLA = "https://raw.githubusercontent.com/JuanFKurucz/ucu-la-project/main/"
for archivo in ["corpus/langchain__short-term-memory.md", "corpus/langgraph__interrupts.md", "corpus/langgraph__persistence.md",
                "corpus/langgraph__thinking-in-langgraph.md", "corpus/langgraph__use-subgraphs.md",
                "datos/m6_seleccion.jsonl", "fixtures/web.json"]:
    Path(archivo).parent.mkdir(exist_ok=True)
    urllib.request.urlretrieve(PLANTILLA + archivo, archivo)
seleccion = [json.loads(linea) for linea in open("datos/m6_seleccion.jsonl", encoding="utf-8") if linea.strip()]

cortador = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)
coleccion = chromadb.Client().get_or_create_collection("corpus_1000_150")
for archivo in sorted(Path("corpus").glob("*.md")):
    partes = cortador.split_text(archivo.read_text(encoding="utf-8"))
    coleccion.upsert(ids=[f"{archivo.stem}#{i}" for i in range(len(partes))], documents=partes,
                     metadatas=[{"doc": archivo.stem}] * len(partes))
os.environ["WEB"] = "grabada"  # "grabada": siempre la misma respuesta · "caida": devuelve un ERROR


@tool
def buscar_documentos(consulta: str) -> str:
    """Busca en la documentación del corpus (LangGraph/LangChain: persistencia, interrupts,
    subgrafos, memoria). Usar para preguntas técnicas sobre esos temas."""
    r = coleccion.query(query_texts=[consulta], n_results=3)
    return "\n\n".join(f"[{m['doc']}] {t[:600]}" for m, t in zip(r["metadatas"][0], r["documents"][0]))


@tool
def buscar_web(consulta: str) -> str:
    """Busca en internet información reciente o que no está en la documentación del corpus
    (noticias, precios, versiones nuevas, empresas, fechas)."""
    if os.environ["WEB"] == "caida":
        return "ERROR: la búsqueda web no respondió. No inventes; decíselo al usuario."
    return json.loads(Path("fixtures/web.json").read_text(encoding="utf-8"))["resultado"]


@tool
def calcular(operacion: Literal["suma", "resta", "multiplicacion", "division", "porcentaje_cambio"],
             a: float, b: float) -> str:
    """Hace una cuenta con dos números. porcentaje_cambio = (b - a) / a * 100. Usar para cualquier cuenta."""
    resultado = {"suma": a + b, "resta": a - b, "multiplicacion": a * b, "division": a / b if b else None,
                 "porcentaje_cambio": (b - a) / a * 100 if a else None}[operacion]
    return "ERROR: división por cero. No inventes el resultado." if resultado is None else f"{round(resultado, 4) + 0.0:.12g}"


SISTEMA = ("Sos un asistente de investigación. Usá buscar_documentos para LangGraph/LangChain, "
           "buscar_web para información reciente o externa y calcular para cualquier cuenta. "
           "Si una herramienta devuelve ERROR, decilo y no inventes. Respondé en español, breve.")


def topes():  # los límites van en el código, no en el prompt
    return [ModelCallLimitMiddleware(run_limit=6), ToolCallLimitMiddleware(run_limit=5)]


agente = create_agent(modelo, tools=[buscar_documentos, buscar_web, calcular], system_prompt=SISTEMA,
                      middleware=topes(), checkpointer=InMemorySaver())  # tu v0.4


class Contador(BaseCallbackHandler):
    """Cuenta las llamadas al modelo: LangChain avisa cada vez que empieza una."""
    def __init__(self):
        self.llamadas = 0

    def on_chat_model_start(self, serialized, messages, **kwargs):
        self.llamadas += 1


def preguntar(pregunta):
    contador = Contador()
    config = {"configurable": {"thread_id": uuid.uuid4().hex}, "callbacks": [contador]}
    mensajes = agente.invoke({"messages": [("user", pregunta)]}, config)["messages"]
    return mensajes, contador.llamadas


def pedidas(mensajes):
    return [p["name"] for m in mensajes for p in getattr(m, "tool_calls", [])]


def mostrar(mensajes):
    for m in mensajes:
        if m.type == "human":
            print(f"[pregunta]     {m.text}")
        elif m.type == "ai" and m.tool_calls:
            for pedido in m.tool_calls:
                print(f"[modelo pide]  {pedido['name']}({pedido['args']})")
        elif m.type == "tool":
            print(f"[tu código]    {m.name} → {m.text[:90]!r}")
        else:
            print(f"[respuesta]    {m.text}")


print(f"{coleccion.count()} fragmentos · {len(seleccion)} preguntas de selección · tu v0.4 lista")

# Parte A · El equipo con una persona que firma

## 1 · El equipo, paso a paso (slides 14 a 16)

Tu v0.5: el supervisor elige quién sigue, el investigador busca, el analista calcula, el redactor escribe y una persona
firma. Acá cada caja imprime lo que **devuelve**, como en la slide 15. **Predecí:** con la pregunta de la tabla × 3,
¿el supervisor llama al analista para la cuenta?

In [ ]:
import operator
from typing import Annotated, TypedDict

from langgraph.graph import END, START, StateGraph
from langgraph.types import Command, interrupt
from pydantic import BaseModel

INVESTIGADOR = ("Investigás: buscar_documentos para LangGraph/LangChain, buscar_web para datos externos "
                "o recientes. Devolvé hallazgos breves con su fuente. Si una herramienta da ERROR, decilo.")
SUPERVISOR = ("Elegí el próximo paso. investigador: falta información. analista: hay que calcular "
              "con lo hallado. redactor: ya alcanza.")


class Ruta(BaseModel):
    """Molde para la decisión del supervisor: solo puede elegir una de estas tres."""
    siguiente: Literal["investigador", "analista", "redactor"]


class Estado(TypedDict, total=False):
    pregunta: str
    notas: Annotated[list[str], operator.add]  # cada especialista SUMA notas, no las pisa
    pasos: int
    siguiente: str
    borrador: str
    aprobada: bool
    respuesta_final: str


def armar_equipo(instrucciones_supervisor):
    investigador = create_agent(modelo, tools=[buscar_documentos, buscar_web], system_prompt=INVESTIGADOR, middleware=topes())
    analista = create_agent(modelo, tools=[calcular], system_prompt="Hacés cuentas con calcular sobre los hallazgos. Breve.",
                            middleware=topes())

    def supervisor(estado):
        if estado.get("pasos", 0) >= 3:  # tope en código: no depende del modelo
            return {"siguiente": "redactor"}
        ruta = modelo.with_structured_output(Ruta).invoke(
            f"{instrucciones_supervisor}\nPregunta: {estado['pregunta']}\nNotas: {estado.get('notas') or 'ninguna'}")
        return {"siguiente": ruta.siguiente if ruta else "redactor", "pasos": estado.get("pasos", 0) + 1}

    def especialista(agente_):
        def nodo(estado):
            mensajes = agente_.invoke({"messages": [("user", f"{estado['pregunta']}\nNotas previas: {estado.get('notas') or 'ninguna'}")]})["messages"]
            return {"notas": [mensajes[-1].text]}
        return nodo

    def redactor(estado):
        return {"borrador": modelo.invoke(f"Respondé '{estado['pregunta']}' en 2 o 3 frases, SOLO con estas notas; "
                                          f"si no alcanzan, decilo.\nNotas: {estado.get('notas') or 'ninguna'}").text}

    def aprobar(estado):
        return {"aprobada": interrupt({"borrador": estado["borrador"]}) is True}  # PAUSA hasta que una persona decide

    g = StateGraph(Estado)
    for nombre, funcion in [("supervisor", supervisor), ("investigador", especialista(investigador)),
                            ("analista", especialista(analista)), ("redactor", redactor), ("aprobar", aprobar),
                            ("final", lambda e: {"respuesta_final": e["borrador"]})]:
        g.add_node(nombre, funcion)
    g.add_edge(START, "supervisor")
    g.add_conditional_edges("supervisor", lambda e: e["siguiente"],
                            {"investigador": "investigador", "analista": "analista", "redactor": "redactor"})
    g.add_edge("investigador", "supervisor")
    g.add_edge("analista", "supervisor")
    g.add_edge("redactor", "aprobar")
    g.add_conditional_edges("aprobar", lambda e: "final" if e["aprobada"] else END, {"final": "final", END: END})
    g.add_edge("final", END)
    return g.compile(checkpointer=InMemorySaver())


def correr_hasta_la_firma(equipo, pregunta):
    """Corre el equipo hasta la pausa e imprime lo que devuelve cada caja; devuelve el hilo y la ruta."""
    hilo = {"configurable": {"thread_id": uuid.uuid4().hex}}
    ruta = []
    for paso in equipo.stream({"pregunta": pregunta}, hilo, stream_mode="updates"):
        for caja, cambio in paso.items():
            if caja == "__interrupt__":
                print(f"[PAUSA]        espera a una persona · borrador: {cambio[0].value['borrador'][:110]!r}")
            else:
                ruta.append(caja)
                print(f"[{caja}]".ljust(15), str(cambio)[:110])
    return hilo, ruta


PREGUNTA = ("Según la documentación, ¿cuántas filas tiene la tabla que compara checkpointer y store? "
            "Multiplicá ese número por 3.")
equipo = armar_equipo(SUPERVISOR)
hilo, ruta = correr_hasta_la_firma(equipo, PREGUNTA)
print("\nLa ruta:", " → ".join(ruta), "· ¿llamó al analista?", "analista" in ruta)

**Mirá:** si el analista no aparece en la ruta, la cuenta la hizo el redactor **sin calculadora**: puede salir bien,
pero nadie la comprobó (slide 15). ¿Cómo harías que pida al analista cuando hay una cuenta?

## 2 · El grafo se pausa y retoma (slide 17) · sin gastar más

El grafo quedó **en pausa** en `aprobar`. El checkpointer guarda el estado del hilo, como el paso «Iniciar y esperar
una aprobación» de Power Automate. **Predecí:** ¿qué hay guardado? Después decidí vos: `True` aprueba, `False` rechaza.

In [ ]:
pausa = equipo.get_state(hilo)
print("Le toca a:", pausa.next, "· guardado:", list(pausa.values), "\n")

DECISION = True  # tu firma: True aprueba, False rechaza (con False, volvé a correr la sección 1 para otro hilo)
final = equipo.invoke(Command(resume=DECISION), hilo)  # mismo hilo = mismo estado
print("Respuesta final:", final.get("respuesta_final", "(ninguna: el rechazo la bloqueó)"))

## 3 · ¿Qué cambiarías en el supervisor? (slide 19) · gasta unos 6 pedidos, opcional

Tres perillas: el **modelo**, las **instrucciones** o el **tope de pasos**. Acá cambiamos solo las instrucciones.
**Predecí:** si le decís que toda cuenta va al analista, ¿lo llama?

In [ ]:
OTRAS = SUPERVISOR + " Si la pregunta pide una cuenta, el analista la hace antes del redactor."  # cambialas vos
hilo2, ruta2 = correr_hasta_la_firma(armar_equipo(OTRAS), PREGUNTA)
print("\nAntes:", " → ".join(ruta), "\nAhora:", " → ".join(ruta2))

**Mirá:** cambiaste **una sola cosa**. Para saber si mejoró de verdad, la medida es la suite entera (práctica M7), no una
pregunta.

# Parte B · Producción: traza, costo, caché y guardrails

## 4 · Cada pregunta deja una línea en la traza (slides 7 a 9)

Tu v1.0 envuelve tu versión de siempre (acá, el agente v0.4): **tapa los emails**, calcula el **costo** y deja una línea
por pregunta en `logs/traza.jsonl`, como el historial de actualizaciones de Power BI. **Predecí:** ¿aparece el email en la traza?

In [ ]:
import datetime
import re

PRECIO_ENTRADA, PRECIO_SALIDA, PRECIO_FECHA = 0.30, 2.50, "19/10"  # US$ por millón de tokens, de la página del proveedor


def tapar(texto):  # el guardrail: un email no llega al modelo ni a la traza
    return re.sub(r"[\w.+-]+@[\w-]+(\.[\w-]+)+", "[email]", texto)


def responder_v04(pregunta):
    mensajes, llamadas = preguntar(pregunta)
    uso = [m.usage_metadata for m in mensajes if getattr(m, "usage_metadata", None)]
    return {"respuesta": mensajes[-1].text, "herramientas": pedidas(mensajes), "llamadas": llamadas,
            "tokens_entrada": sum(u["input_tokens"] for u in uso), "tokens_salida": sum(u["output_tokens"] for u in uso)}


base = responder_v04  # en tu repo: v05_equipo.responder o v04_agente.responder


def responder(pregunta):  # tu v1.0
    try:
        r = base(tapar(pregunta))
    except Exception as error:  # fallo seguro: un aviso claro, nunca una respuesta inventada
        r = {"respuesta": "No pude responder ahora. Probá en unos minutos.", "error": type(error).__name__,
             "tokens_entrada": 0, "tokens_salida": 0}
    r["costo_usd"] = (r["tokens_entrada"] * PRECIO_ENTRADA + r["tokens_salida"] * PRECIO_SALIDA) / 1_000_000
    os.makedirs("logs", exist_ok=True)
    with open("logs/traza.jsonl", "a", encoding="utf-8") as traza:  # una línea por pregunta
        fila = {"cuando": datetime.datetime.now().isoformat(timespec="seconds"), "pregunta": tapar(pregunta),
                "precio_del": PRECIO_FECHA, **r}
        traza.write(json.dumps(fila, ensure_ascii=False) + "\n")
    return r


print(responder("¿Qué hace interrupt()? Respondeme a ana.perez@ucu.edu.uy")["respuesta"], "\n")
print(open("logs/traza.jsonl", encoding="utf-8").read().splitlines()[-1])

**Mirá:** en la traza dice `[email]`: el dato personal no llegó ni al modelo ni al registro (Ley 18.331). Langfuse o
LangSmith guardan además **cada paso** de la consulta; tu traza, una línea por pregunta.

## 5 · Del token al mes: multiplicá y proyectá (slide 10) · sin gastar

La traza es una tabla: la leemos con pandas y proyectamos el mes. **Predecí:** a 1000 preguntas por día, ¿cuánto por mes?
Probá con los precios de otro modelo.

In [ ]:
traza = pd.read_json("logs/traza.jsonl", lines=True)
ultima = traza.iloc[-1]
print(f"Por consulta: {ultima['tokens_entrada']} tokens de entrada, {ultima['tokens_salida']} de salida, US$ {ultima['costo_usd']:.6f}")
print(f"Por mes, a 1000 por día: US$ {ultima['costo_usd'] * 1000 * 30:.2f} (precios del {PRECIO_FECHA})")
traza[["cuando", "pregunta", "herramientas", "llamadas", "tokens_entrada", "tokens_salida", "costo_usd"]]

## 6 · Una caché responde lo repetido sin el modelo (slides 11 y 12) · 2 pedidos

La caché guarda por el **texto exacto** del pedido. **Predecí:** la misma pregunta dos veces, y una tercera con una
minúscula: ¿cuáles llegan al modelo?

In [ ]:
from langchain_core.caches import InMemoryCache
from langchain_core.globals import set_llm_cache

set_llm_cache(InMemoryCache())  # guarda por el texto exacto del pedido
for pregunta in ("¿Qué hace interrupt() en LangGraph? En una frase.", "¿Qué hace interrupt() en LangGraph? En una frase.",
                 "¿qué hace interrupt() en LangGraph? En una frase."):
    inicio = time.perf_counter()
    r = modelo.invoke(pregunta)
    print(f"{time.perf_counter() - inicio:.2f} s · tokens {r.usage_metadata['total_tokens']} · «{pregunta[:26]}…»")
set_llm_cache(None)  # apagada para el resto

**Mirá:** la guardada tarda casi 0 y trae **sus tokens viejos**: si sumás el costo con esos números, contás de más.
Una minúscula distinta ya es otro pedido. La caché **semántica** compara por parecido: ahorra más, pero puede devolver
la respuesta de otra pregunta.

## 7 · Guardrails en código: tapar y no inventar (slides 13, 15 y 16) · sin gastar

Un **guardrail** es un control en código, como una baranda. **Predecí:** ¿cuáles de estos tapa? Agregá un caso tuyo.
Después simulamos que el modelo se cae.

In [ ]:
for texto in ["Mi mail es ana.perez@ejemplo.com", "Escribile a juan@ucu.edu.uy y a maría.lópez@empresa.com.uy",
              "Mi cédula es 1.234.567-8", "Agregá acá un caso tuyo"]:
    print(f"{texto!r:62} → {tapar(texto)!r}")


def se_cae(pregunta):  # simula que el modelo o la red se caen
    raise ConnectionError("el servicio no respondió")


base = se_cae
print("\nCon el modelo caído:", responder("¿Qué hace interrupt()?"))
base = responder_v04  # de vuelta

**Mirá:** el email se tapa, pero **la cédula no**: el guardrail ataja solo lo que previste (slide 14). Con el modelo
caído, la persona ve un aviso honesto y la traza guarda el `error`. Si responde de memoria sin buscar, no lo ataja
ninguno de los dos: eso lo mide la suite.

## 8 · Análisis de errores: una tabla dinámica de tus fallas (slides 17 a 20) · sin gastar

Antes de medir más, **leé** las respuestas que fallaron y anotá el **primer error** de cada una. Después contás por tipo,
como una tabla dinámica de Excel o Power BI. Estas son 4 de las 9 fallas del modelo chico en v0.5 (slide 19);
agregá filas con las tuyas.

In [ ]:
fallas = pd.DataFrame([
    {"id": 1, "pregunta": "¿Qué hace interrupt() en LangGraph?", "usó": "nada", "esperaba": "buscar_documentos", "primer_error": "no buscó en los documentos"},
    {"id": 3, "pregunta": "¿Qué clase borra mensajes del estado?", "usó": "calcular", "esperaba": "buscar_documentos", "primer_error": "pidió una herramienta equivocada"},
    {"id": 5, "pregunta": "¿Última versión de LangGraph en PyPI?", "usó": "calcular ×3", "esperaba": "buscar_web", "primer_error": "pidió una herramienta equivocada"},
    {"id": 11, "pregunta": "Gracias, eso es todo.", "usó": "buscar_documentos", "esperaba": "nada", "primer_error": "usó una herramienta sin necesidad"},
    # agregá acá tus fallas: {"id": …, "pregunta": "…", "usó": "…", "esperaba": "…", "primer_error": "…"},
])
display(fallas)
fallas.pivot_table(index="primer_error", values="id", aggfunc="count").rename(columns={"id": "fallas"}).sort_values("fallas", ascending=False)

**Mirá:** la tabla dinámica dice **qué arreglar primero**: el tipo más frecuente. En la corrida entera del modelo chico
fueron 4 «no buscó», 4 «equivocada» y 1 «de más»: ninguna «buscó bien y respondió mal». En el taller hacés lo mismo con
tu `logs/medir.csv` (práctica M8).